# EPO Patent Search - Simplified Version

**Features:**
- Patent bibliographic information (title, abstract, applicants, inventors)
- CPC codes and classification
- Deep-tech domain classification (CPC to domain mapping)
- Strict fuzzy name validation (95% minimum match)
- Multi-sheet Excel output (Summary, Patent Details, CPC Reference, Deep-Tech Analysis)
- File-based caching (50-90% faster on repeated runs)
- Exponential backoff retry (automatic error recovery)
- Checkpoint system (resume capability)
- Modular architecture (maintainable code)
- Configuration management (easy parameter tuning)

**Note:** Patent family and legal status features have been removed from this version.

**Date:** 2025-11-28

## 1. Import Modules

In [1]:
# Configuration and setup
from config.settings import load_config
from utils.logging_config import setup_logging
from utils.country_codes import COUNTRY_CODE_MAP
from utils.namespaces import BIBLIO_NAMESPACES

# Core functionality
from core.api_client import EPOClient

# Data I/O
from data.input_reader import read_company_names_from_excel
from data.output_writer import save_results_to_excel
from data.checkpoint_manager import CheckpointManager

# Processing
from processors.company_processor import clean_company_name_for_epo, search_and_process_company
from processors.deeptech_classifier import DeepTechClassifier  # NEW

# Standard library
import pandas as pd
from datetime import datetime
import time
import logging

print("✓ All modules imported successfully")

✓ All modules imported successfully


## 2. Load Configuration and Initialize

In [2]:
# === INPUT FILE CONFIGURATION ===
# Set to filename to override config.json, or leave as None to use config default
INPUT_FILE = "2026 DT Startup List 3.xlsx"  # Change this as needed, or set to None

# Alternatively, you can specify full/relative paths:
# INPUT_FILE = "./data/my_companies.xlsx"
# INPUT_FILE = None  # Uses config.json default

# Load configuration
print("Loading configuration...")
config = load_config()

# Override input file if specified
if INPUT_FILE is not None:
    config['input']['excel_file'] = INPUT_FILE
    print(f"  📁 Input file: {INPUT_FILE} (override)")
else:
    print(f"  📁 Input file: {config['input']['excel_file']} (from config)")

# Setup logging
setup_logging(config)
logging.info("=" * 60)
logging.info("EPO Patent Search - Refactored Version")
logging.info("=" * 60)

# Initialize EPO API client
logging.info("Initializing EPO API client...")
epo_client = EPOClient(config)
logging.info("✓ EPO client initialized")

# Initialize checkpoint manager
logging.info("Initializing checkpoint manager...")
checkpoint_mgr = CheckpointManager(config['checkpoint']['checkpoint_dir'])
logging.info("✓ Checkpoint manager initialized")

# Initialize deep-tech classifier (NEW)
deeptech_classifier = None
if config.get('deeptech', {}).get('enabled', False):
    deeptech_settings = config['deeptech'].get('settings')
    if deeptech_settings:
        deeptech_classifier = DeepTechClassifier(
            deeptech_settings['mapping_file'],
            deeptech_settings
        )
        logging.info("✓ Deep-tech classifier initialized")
        print("  - Deep-tech classification: ENABLED")
    else:
        logging.warning("Deep-tech enabled but configuration failed to load")
        print("  - Deep-tech classification: DISABLED (config error)")
else:
    print("  - Deep-tech classification: DISABLED")

print("\n✓ Configuration loaded and systems initialized")
print(f"  - Cache enabled: {config['cache']['enabled']}")
print(f"  - Checkpoints enabled: {config['checkpoint']['enabled']}")
print(f"  - Max results per company: {config['api']['max_results_per_company']}")
print(f"  - Retry attempts: {config['retry']['max_attempts']}")

2026-03-15 20:33:02 - INFO - Logging configured with level: INFO
2026-03-15 20:33:02 - INFO - ============================================================
2026-03-15 20:33:02 - INFO - EPO Patent Search - Refactored Version
2026-03-15 20:33:02 - INFO - ============================================================
2026-03-15 20:33:02 - INFO - Initializing EPO API client...
2026-03-15 20:33:02 - INFO - Cache manager initialized
2026-03-15 20:33:02 - INFO - ✓ EPO client initialized
2026-03-15 20:33:02 - INFO - Initializing checkpoint manager...
2026-03-15 20:33:02 - INFO - ✓ Checkpoint manager initialized
2026-03-15 20:33:02 - INFO - DeepTechClassifier initialized with 58 CPC main groups mapped
2026-03-15 20:33:02 - INFO - ✓ Deep-tech classifier initialized


Loading configuration...
  📁 Input file: 2026 DT Startup List 3.xlsx (override)
  - Deep-tech classification: ENABLED

✓ Configuration loaded and systems initialized
  - Cache enabled: True
  - Checkpoints enabled: True
  - Max results per company: 10
  - Retry attempts: 3


## 2.5. Configure Strict Validation Mode

This notebook uses **strict validation mode** to ensure only exact company name matches are flagged as "Validated".

**Strict Mode Features:**
- **Exact token matching**: Requires 95%+ of company name tokens to appear in the applicant name
- **Prevents false positives**: "Alfred Pte. Ltd." will NOT match inventor names like "Dr. Alfred Smith"
- **Configurable thresholds**: 
  - ≥95% = Validated (exact match)
  - 70-94% = Uncertain (partial match, review recommended)
  - <70% = Failed (no match)
- **All results included**: Even failed matches are kept in the output for manual review

**Usage:**
- Results are flagged but NOT excluded from the output
- Filter Excel by `validation_status = "Validated"` to see only exact matches
- Review "Uncertain" results manually to catch legitimate subsidiaries/variations

# === STRICT MATCHING CONFIGURATION OVERRIDE ===
# Override config to enable strict validation mode
print("\n⚙ Applying strict matching configuration overrides...")

config['search_validation'] = config.get('search_validation', {})
config['search_validation']['enabled'] = True
config['search_validation']['validation_mode'] = 'strict'  # NEW: Use strict token matching
config['search_validation']['min_similarity_score'] = 95   # Require 95% token overlap
config['search_validation']['uncertain_threshold'] = 70    # 70-94% = Uncertain
config['search_validation']['exclude_failed_validations'] = False  # Flag but don't exclude

# Optional: Test different CQL operators
config['search'] = config.get('search', {})
config['search']['cql_operator'] = 'pa'  # Change to 'pa_name' or 'apn' if testing applicant-only

# Disable aggressive suffix removal for strict searches (NEW)
config['name_normalization'] = config.get('name_normalization', {})
config['name_normalization']['remove_legal_suffixes'] = False  # Don't create "ALFRED" from "ALFRED PTE. LTD."

print("  ✓ Strict validation mode: ENABLED")
print(f"  ✓ Validation thresholds: {config['search_validation']['min_similarity_score']}% / {config['search_validation']['uncertain_threshold']}%")
print(f"  ✓ CQL operator: {config['search']['cql_operator']}")
print(f"  ✓ Exclude failed validations: {config['search_validation']['exclude_failed_validations']}")
print(f"  ✓ Legal suffix removal: DISABLED (strict search mode)")

## 3. Check for Existing Checkpoints (Resume Capability)

In [3]:
# Check for existing checkpoints
summary_df = None
all_patent_details = []
start_index = 0

if checkpoint_mgr.is_checkpoint_available() and config['checkpoint']['auto_resume']:
    print("\n⚠ Existing checkpoint found!")
    
    # List available checkpoints
    checkpoints = checkpoint_mgr.list_checkpoints()
    if checkpoints:
        latest = checkpoints[-1]
        print(f"  Latest checkpoint: {latest['filename']}")
        print(f"  Run ID: {latest['metadata']['run_id']}")
        print(f"  Last processed: index {latest['metadata']['last_processed_index']}")
        print(f"  Checkpoint time: {latest['metadata']['checkpoint_timestamp']}")
    
    user_input = input("\nResume from checkpoint? (yes/no): ").strip().lower()
    
    if user_input in ['yes', 'y']:
        logging.info("Loading checkpoint...")
        checkpoint_data = checkpoint_mgr.load_checkpoint()
        
        if checkpoint_data:
            summary_df = checkpoint_data['summary_df']
            all_patent_details = checkpoint_data['details_list']
            start_index = checkpoint_data['last_index'] + 1
            
            print(f"\n✓ Checkpoint loaded successfully")
            print(f"  Resuming from index {start_index}")
            print(f"  Already processed: {len(all_patent_details)} patent details")
        else:
            print("\n✗ Failed to load checkpoint, starting fresh")
    else:
        print("\n→ Starting fresh (checkpoint ignored)")
else:
    print("\n→ No checkpoints found, starting fresh")


⚠ Existing checkpoint found!
  Latest checkpoint: checkpoint_20260312_203215_7c152931_915.json
  Run ID: 20260312_203215_7c152931
  Last processed: index 4574
  Checkpoint time: 2026-03-13T23:37:18.213178

Resume from checkpoint? (yes/no): yes


2026-03-15 20:34:52 - INFO - Loading checkpoint...
2026-03-15 20:34:52 - INFO - Loading checkpoint from: checkpoint_20260312_203215_7c152931_915.json



✓ Checkpoint loaded successfully
  Resuming from index 4575
  Already processed: 2473 patent details


## 4. Load Input Data

In [4]:
# Read company names from Excel
logging.info("Reading company names from Excel...")
companies = read_company_names_from_excel(
    config['input']['excel_file'],
    config['input']['sheet_name'],
    config['input']['column_name']
)

if not companies:
    print("\n✗ No company names found in input file. Exiting.")
    raise ValueError("No companies to process")

print(f"\n✓ Loaded {len(companies)} companies from input file")

# Initialize or verify summary DataFrame
if summary_df is None:
    logging.info("Initializing summary DataFrame...")
    summary_df = pd.DataFrame({
        'Original Name': companies,
        'Cleaned Name (for Query)': [clean_company_name_for_epo(name) for name in companies],
        'Status': 'Not Processed',
        'Search Error Message': pd.NA,
        'API Total Results': pd.NA,
        'Details Processed Count': 0,
        'Processed Country Distribution': pd.NA,
        'Processed Timestamp': pd.NaT
    })
    summary_df.index.name = 'Company Index'
    print("✓ Summary DataFrame initialized")
else:
    print("✓ Using existing summary DataFrame from checkpoint")

print(f"\nProcessing plan:")
print(f"  Total companies: {len(companies)}")
print(f"  Starting from index: {start_index}")
print(f"  Companies remaining: {len(companies) - start_index}")

2026-03-15 20:34:56 - INFO - Reading company names from Excel...
2026-03-15 20:34:59 - INFO - Successfully read 4697 company names from '2026 DT Startup List 3.xlsx' (Sheet: 'Sheet1', Column: 'Coy Name')



✓ Loaded 4697 companies from input file
✓ Using existing summary DataFrame from checkpoint

Processing plan:
  Total companies: 4697
  Starting from index: 4575
  Companies remaining: 122


## 5. Main Processing Loop

In [5]:
# Main processing loop
processed_count = start_index
start_time = time.time()

logging.info("\nStarting main processing loop...")
logging.info("=" * 60)

for index, row in summary_df.iloc[start_index:].iterrows():
    original_name = row['Original Name']
    cleaned_name = row['Cleaned Name (for Query)']
    processed_count += 1
    
    logging.info(f"\n{'='*60}")
    logging.info(f"Processing {processed_count}/{len(summary_df)}: {original_name}")
    logging.info(f"{'='*60}")
    
    if not cleaned_name:
        logging.warning(f"Skipping processing for index {index} as cleaned name is empty")
        summary_df.loc[index, 'Status'] = 'Skipped - Empty Cleaned Name'
        summary_df.loc[index, 'Processed Timestamp'] = datetime.now()
        continue
    
    # Search and process company
    try:
        status, error_msg, api_count, details_list = search_and_process_company(
            original_name,
            cleaned_name,
            epo_client,
            BIBLIO_NAMESPACES,
            COUNTRY_CODE_MAP,
            config['api']['max_results_per_company'],
            config['api']['search_delay_seconds'],
            config['api']['detail_delay_seconds'],
            deeptech_classifier,
            config  # Pass config for validation and name normalization
        )
    except Exception as process_err:
        logging.error(f"!! UNEXPECTED ERROR processing company '{original_name}': {process_err}", exc_info=True)
        status = "Processing Error - Unexpected"
        error_msg = str(process_err)
        api_count = 'N/A'
        details_list = []
    
    # Update summary
    summary_df.loc[index, 'Status'] = status
    summary_df.loc[index, 'Search Error Message'] = error_msg if error_msg else pd.NA
    summary_df.loc[index, 'API Total Results'] = api_count
    summary_df.loc[index, 'Details Processed Count'] = len(details_list)
    summary_df.loc[index, 'Processed Timestamp'] = datetime.now()
    
    if details_list:
        all_patent_details.extend(details_list)
    
    # Checkpoint save
    if config['checkpoint']['enabled'] and processed_count % config['checkpoint']['checkpoint_interval'] == 0:
        logging.info(f"\nSaving checkpoint at index {index}...")
        try:
            checkpoint_mgr.save_checkpoint(
                summary_df,
                all_patent_details,
                index,
                len(summary_df),
                statistics={
                    'processed_count': processed_count,
                    'total_details': len(all_patent_details),
                    'cache_stats': epo_client.get_cache_stats()
                }
            )
        except Exception as checkpoint_err:
            logging.error(f"Failed to save checkpoint: {checkpoint_err}")
    
    # Delay between companies
    if processed_count < len(summary_df):
        logging.debug(f"\n(Waiting {config['api']['search_delay_seconds']}s before next company search...)")
        time.sleep(config['api']['search_delay_seconds'])

end_time = time.time()
logging.info("\n" + "="*60)
logging.info("Main processing loop completed")
logging.info("="*60)

print(f"\n✓ Processing completed in {end_time - start_time:.2f} seconds")

2026-03-15 20:35:02 - INFO - 
Starting main processing loop...
2026-03-15 20:35:02 - INFO - ============================================================
2026-03-15 20:35:02 - INFO - 
2026-03-15 20:35:02 - INFO - Processing 4576/4697: F. E. Solution
2026-03-15 20:35:02 - INFO - ============================================================
2026-03-15 20:35:02 - INFO - ------------------------------------------------------------
2026-03-15 20:35:02 - INFO - Multi-variant search enabled
2026-03-15 20:35:02 - INFO - NameNormalizer initialized with advanced normalization
2026-03-15 20:35:02 - INFO - Generated 1 search variants for 'F. E. Solution'
2026-03-15 20:35:02 - INFO - Trying variant 1/1: 'F. E. SOLUTION' (Original: 'F. E. Solution')
2026-03-15 20:35:02 - INFO - Using CQL Query: pa="F. E. SOLUTION"
2026-03-15 20:35:02 - INFO - Cache MISS for search query: pa="F. E. SOLUTION"
2026-03-15 20:35:14 - WARNING - Attempt 1/3 failed with ReadTimeout. Retrying in 2.65s...
2026-03-15 20:35:17 - 

2026-03-15 20:35:36 - INFO - 
2026-03-15 20:35:36 - INFO - Processing 4578/4697: [24]7.ai, Inc.
2026-03-15 20:35:36 - INFO - ============================================================
2026-03-15 20:35:36 - INFO - ------------------------------------------------------------
2026-03-15 20:35:36 - INFO - Multi-variant search enabled
2026-03-15 20:35:36 - INFO - NameNormalizer initialized with advanced normalization
2026-03-15 20:35:36 - INFO - Generated 1 search variants for '[24]7.ai, Inc.'
2026-03-15 20:35:36 - INFO - Trying variant 1/1: '7.AI INC.' (Original: '[24]7.ai, Inc.')
2026-03-15 20:35:36 - INFO - Using CQL Query: pa="7.AI INC."
2026-03-15 20:35:36 - INFO - Cache HIT for search query: pa="7.AI INC."
2026-03-15 20:35:36 - INFO - Search successful for '7.AI INC.'
2026-03-15 20:35:36 - INFO - Total results reported by API: 159
2026-03-15 20:35:36 - INFO - Found 10 publication references in search results (up to max 10) using variant '7.AI INC.'
2026-03-15 20:35:36 - INFO - --- E

2026-03-15 20:36:00 - INFO -     Retrieving bibliographic details...
2026-03-15 20:36:00 - INFO - Cache HIT for biblio: NL2027691B1
2026-03-15 20:36:00 - INFO -     Found 2 unique CPC codes: G06F 18/285, G06V 40/172
2026-03-15 20:36:00 - INFO -     Deep-tech Classification: Moderate (Primary: Unclassified, Score: 50)
2026-03-15 20:36:00 - INFO -     Biblio details retrieved successfully
2026-03-15 20:36:02 - INFO -   Processing Publication: EP3905120A1 (European Patent Office)
2026-03-15 20:36:02 - INFO -     Retrieving bibliographic details...
2026-03-15 20:36:02 - INFO - Cache HIT for biblio: EP3905120A1
2026-03-15 20:36:02 - INFO -     Found 7 unique CPC codes: G06V 10/454, G06V 10/764, G06V 10/82, G06V 10/993, G06V 40/161, G06V 40/168, G06V 40/172
2026-03-15 20:36:02 - INFO -     Deep-tech Classification: Very Strong (Primary: AI/ML, Score: 85)
2026-03-15 20:36:02 - INFO -     Biblio details retrieved successfully
2026-03-15 20:36:04 - INFO -   Processing Publication: NL2025430B1 (

2026-03-15 20:36:57 - INFO - 
2026-03-15 20:36:57 - INFO - Processing 4584/4697: Anaut
2026-03-15 20:36:57 - INFO - ============================================================
2026-03-15 20:36:57 - INFO - ------------------------------------------------------------
2026-03-15 20:36:57 - INFO - Multi-variant search enabled
2026-03-15 20:36:57 - INFO - NameNormalizer initialized with advanced normalization
2026-03-15 20:36:57 - INFO - Generated 1 search variants for 'Anaut'
2026-03-15 20:36:57 - INFO - Trying variant 1/1: 'ANAUT' (Original: 'Anaut')
2026-03-15 20:36:57 - INFO - Using CQL Query: pa="ANAUT"
2026-03-15 20:36:57 - INFO - Cache MISS for search query: pa="ANAUT"
2026-03-15 20:37:12 - WARNING - Attempt 1/3 failed with ConnectionError. Retrying in 2.37s...
2026-03-15 20:37:18 - WARNING - Attempt 2/3 failed with ConnectionError. Retrying in 4.62s...
2026-03-15 20:37:27 - ERROR - All 3 retry attempts exhausted for ConnectionError
2026-03-15 20:37:27 - ERROR - Unexpected error dur

2026-03-15 20:37:54 - INFO -     Found 6 unique CPC codes: H01R 13/506, H01R 13/645, H01R 13/6453, H01R 2103/00, H01R 2201/26, Y02E 60/10
2026-03-15 20:37:54 - INFO -     Deep-tech Classification: Very Strong (Primary: Energy, Score: 85)
2026-03-15 20:37:54 - INFO -     Biblio details retrieved successfully
2026-03-15 20:37:56 - INFO -   Processing Publication: WO2025252095A1 (World Intellectual Property Organization (WIPO))
2026-03-15 20:37:56 - INFO -     Retrieving bibliographic details...
2026-03-15 20:37:56 - INFO - Cache MISS for biblio: WO2025252095A1
2026-03-15 20:38:02 - INFO -     Found 5 unique CPC codes: G05B 19/042, G05B 19/0423, G05B 2219/25257, G05D 1/46, G05D 1/87
2026-03-15 20:38:02 - INFO -     Deep-tech Classification: Very Strong (Primary: Robotics, Score: 95)
2026-03-15 20:38:02 - INFO -     Biblio details retrieved successfully
2026-03-15 20:38:04 - INFO -   Processing Publication: WO2025252096A1 (World Intellectual Property Organization (WIPO))
2026-03-15 20:38:0

2026-03-15 20:38:56 - INFO - Using CQL Query: pa="BRAINSPACE"
2026-03-15 20:38:56 - INFO - Cache MISS for search query: pa="BRAINSPACE"
2026-03-15 20:39:00 - INFO - Search successful for 'BRAINSPACE'
2026-03-15 20:39:00 - INFO - Total results reported by API: 33
2026-03-15 20:39:00 - INFO - Found 10 publication references in search results (up to max 10) using variant 'BRAINSPACE'
2026-03-15 20:39:00 - INFO - --- Extracting Publication Number Components ---
2026-03-15 20:39:00 - INFO - 
--- Processing 10 Found Publication(s) ---
2026-03-15 20:39:00 - INFO -   Processing Publication: CA3245026A1 (Canada)
2026-03-15 20:39:00 - INFO -     Retrieving bibliographic details...
2026-03-15 20:39:00 - INFO - Cache MISS for biblio: CA3245026A1
2026-03-15 20:39:02 - INFO -     Found 24 unique CPC codes: A61M 1/0281, A61M 1/3621, A61M 1/3623, A61M 1/70, A61M 1/88, A61M 2027/004, A61M 2039/025, A61M 2039/0282, A61M 2202/0464, A61M 2205/18, A61M 2205/3303, A61M 2205/3306, A61M 2205/3331, A61M 2205/3

... (27 outputs left out)

2026-03-15 21:04:27 - INFO - 
Saving checkpoint at index 4684...
2026-03-15 21:04:28 - INFO - Checkpoint saved: checkpoint_20260312_203215_7c152931_937.json (processed 4685/4697 companies)
2026-03-15 21:04:32 - INFO - 
2026-03-15 21:04:32 - INFO - Processing 4686/4697: Go Safarii
2026-03-15 21:04:32 - INFO - ============================================================
2026-03-15 21:04:32 - INFO - ------------------------------------------------------------
2026-03-15 21:04:32 - INFO - Multi-variant search enabled
2026-03-15 21:04:32 - INFO - NameNormalizer initialized with advanced normalization
2026-03-15 21:04:32 - INFO - Generated 1 search variants for 'Go Safarii'
2026-03-15 21:04:32 - INFO - Trying variant 1/1: 'GO SAFARII' (Original: 'Go Safarii')
2026-03-15 21:04:32 - INFO - Using CQL Query: pa="GO SAFARII"
2026-03-15 21:04:32 - INFO - Cache MISS for search query: pa="GO SAFARII"
2026-03-15 21:04:37 - ERROR - Non-retryable HTTP error 404: 404 Client Error: Not Found for url: htt

2026-03-15 21:06:00 - INFO - ============================================================
2026-03-15 21:06:00 - INFO - ------------------------------------------------------------
2026-03-15 21:06:00 - INFO - Multi-variant search enabled
2026-03-15 21:06:00 - INFO - NameNormalizer initialized with advanced normalization
2026-03-15 21:06:00 - INFO - Generated 1 search variants for 'NOWanders'
2026-03-15 21:06:00 - INFO - Trying variant 1/1: 'NOWANDERS' (Original: 'NOWanders')
2026-03-15 21:06:00 - INFO - Using CQL Query: pa="NOWANDERS"
2026-03-15 21:06:00 - INFO - Cache MISS for search query: pa="NOWANDERS"
2026-03-15 21:06:05 - ERROR - Non-retryable HTTP error 404: 404 Client Error: Not Found for url: https://ops.epo.org/3.2/rest-services/published-data/search
2026-03-15 21:06:05 - INFO - API reported 404: 'No results found' for 'NOWANDERS'
2026-03-15 21:06:09 - INFO - 
2026-03-15 21:06:09 - INFO - Processing 4694/4697: NexPage LLP
2026-03-15 21:06:09 - INFO - =========================

2026-03-15 21:07:04 - INFO -     Deep-tech Classification: Very Strong (Primary: Biotech, Score: 95)
2026-03-15 21:07:04 - INFO -     Biblio details retrieved successfully
2026-03-15 21:07:06 - INFO -   Processing Publication: WO2022216887A1 (World Intellectual Property Organization (WIPO))
2026-03-15 21:07:06 - INFO -     Retrieving bibliographic details...
2026-03-15 21:07:06 - INFO - Cache MISS for biblio: WO2022216887A1
2026-03-15 21:07:08 - INFO -     Found 26 unique CPC codes: A61K 2039/505, A61K 31/436, A61K 39/3955, A61K 45/06, A61K 47/6849, A61P 29/00, A61P 31/00, A61P 35/00, A61P 35/02, A61P 37/02, C07K 16/2803, C07K 16/2809, C07K 16/2812, C07K 2317/24, C07K 2317/31, C07K 2317/52, C07K 2317/56, C07K 2317/565, C07K 2317/732, C07K 2317/734, C07K 2317/76, C07K 2317/92, G01N 2333/70514, G01N 33/532, G01N 33/577, G01N 33/68
2026-03-15 21:07:08 - INFO -     Deep-tech Classification: Very Strong (Primary: Biotech, Score: 95)
2026-03-15 21:07:08 - INFO -     Biblio details retrieved 


✓ Processing completed in 1940.81 seconds


## 6. Post-Processing: Calculate Country Distribution

In [6]:
# Calculate unique publication country distribution
logging.info("Calculating country distribution...")

if all_patent_details:
    details_df_for_stats = pd.DataFrame(all_patent_details)
    
    if 'country_code' in details_df_for_stats.columns and 'original_search_name' in details_df_for_stats.columns:
        country_distribution = details_df_for_stats.groupby('original_search_name')['country_code'].apply(
            lambda x: ", ".join(sorted(list(x.dropna().astype(str).unique())))
        )
        summary_df['Processed Country Distribution'] = summary_df['Original Name'].map(country_distribution).fillna("")
        logging.info("✓ Country distribution calculated")
    else:
        missing_cols = [col for col in ['country_code', 'original_search_name'] if col not in details_df_for_stats.columns]
        logging.warning(f"Required column(s) {missing_cols} not found in detailed data. Cannot calculate distribution.")
        summary_df['Processed Country Distribution'] = "Error: column(s) missing"
else:
    logging.info("No detailed patent data collected, skipping country distribution calculation.")
    summary_df['Processed Country Distribution'] = ""

print("✓ Post-processing complete")

2026-03-15 21:42:15 - INFO - Calculating country distribution...
2026-03-15 21:42:15 - INFO - ✓ Country distribution calculated


✓ Post-processing complete


## 7. Save Results to Excel

In [7]:
import importlib
from data import output_writer
importlib.reload(output_writer)

# Generate output filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"{config['output']['filename_prefix']}{timestamp}.xlsx"
output_path = f"{config['output']['output_dir']}/{output_filename}"

# Save results
logging.info(f"\nSaving results to {output_path}...")
save_results_to_excel(summary_df, all_patent_details, output_path)

# Clean up checkpoints after successful completion
if config['checkpoint']['enabled']:
    logging.info("Cleaning up checkpoints...")
    checkpoint_mgr.cleanup()

print(f"\n✓ Results saved to: {output_filename}")

2026-03-15 21:42:18 - INFO - 
Saving results to C:\Users\Er Jun Hui\Downloads\EPO_Patent_Simplified/epo_patent_search_results_20260315_214218.xlsx...
2026-03-15 21:42:18 - INFO - Attempting to save results to: C:\Users\Er Jun Hui\Downloads\EPO_Patent_Simplified/epo_patent_search_results_20260315_214218.xlsx
2026-03-15 21:42:18 - INFO - Loaded 293 CPC descriptions from config/cpc_descriptions.json
2026-03-15 21:42:18 - INFO - Initialized CPC description lookup for Patent Details
2026-03-15 21:42:18 - INFO - Added CPC descriptions column to Patent Details
2026-03-15 21:42:18 - INFO - Writing Summary sheet...
2026-03-15 21:42:18 - INFO - Writing Patent Details sheet...
2026-03-15 21:42:19 - INFO - Writing CPC Reference sheet...
2026-03-15 21:42:20 - INFO - CPC Reference sheet created with 6744 unique codes
2026-03-15 21:42:20 - INFO - Writing Deep-Tech Analysis sheet...
2026-03-15 21:42:20 - INFO - Deep-Tech Analysis sheet created
2026-03-15 21:42:21 - INFO - Successfully saved results to


✓ Results saved to: epo_patent_search_results_20260315_214218.xlsx


## 8. Display Summary Statistics

In [8]:
# Display final summary
print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"\nTotal time: {end_time - start_time:.2f} seconds")

print("\n--- Status Summary ---")
status_counts = summary_df['Status'].value_counts()
print(status_counts.to_string())

print("\n--- Cache Statistics ---")
cache_stats = epo_client.get_cache_stats()
for key, value in cache_stats.items():
    print(f"  {key}: {value}")

print(f"\n--- Output ---")
print(f"  File: {output_filename}")
print(f"  Location: {config['output']['output_dir']}")
print(f"  Total patents processed: {len(all_patent_details)}")

print("\n" + "="*60)
logging.info("Script finished successfully")
print("✓ Script finished successfully")


PROCESSING COMPLETE

Total time: 1940.81 seconds

--- Status Summary ---
No Results Found             4086
Found - Success               504
Search Error - HTTP 400       102
Search Error - Unexpected       4
Found - Detail Issues           1

--- Cache Statistics ---


2026-03-15 21:42:25 - INFO - Script finished successfully


  hits: 15
  misses: 313
  expired: 0
  errors: 0
  hit_rate: 4.6%
  total_entries: 3216
  total_size_mb: 50.30

--- Output ---
  File: epo_patent_search_results_20260315_214218.xlsx
  Location: C:\Users\Er Jun Hui\Downloads\EPO_Patent_Simplified
  Total patents processed: 2679

✓ Script finished successfully
